# 🎓 Educational Tutorial: Sentinel-2 Satellite Image Inference Pipeline

This notebook serves as a self-contained learning artifact detailing the implementation of a real-world **Sentinel-2 satellite image inference pipeline** using deep learning.

---

## 1. Concepts & Mathematical Intuition

### A. Sentinel-2 Satellite Imagery
Sentinel-2 is an earth observation mission developed by the European Space Agency (ESA) that provides high-resolution multispectral imagery. While our classifier is trained on EuroSAT RGB ($64 \times 64$ patches), actual satellite frames cover large areas ($10980 \times 10980$ pixels at 10m resolution).

### B. Sliding Window & Overlap Stride Math
To classify a large image, we slide a window of size $W \times W$ across it with stride $S$:
* If $S = W$, the patches are **non-overlapping**.
* If $S < W$, the patches **overlap**. Overlapping enables smoother transitions and context preservation at borders.

The number of patches along width ($n_x$) and height ($n_y$) is computed as:
$$n_x = \lfloor \frac{\text{Width} - W}{S} \rfloor + 1$$
$$n_y = \lfloor \frac{\text{Height} - W}{S} \rfloor + 1$$

### C. Image Reconstruction
When patches overlap, a pixel $(x, y)$ can belong to multiple patches. We accumulate the predictions/probabilities at each pixel and divide by the number of visits to compute the average prediction color or confidence.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import time

print("PyTorch version:", torch.__version__)

In [ ]:
class PatchGenerator:
    def __init__(self, patch_size: int = 64, stride: int = 64):
        self.patch_size = patch_size
        self.stride = stride

    def extract_patches(self, image: Image.Image):
        width, height = image.size
        y_idx = 0
        for y in range(0, height - self.patch_size + 1, self.stride):
            x_idx = 0
            for x in range(0, width - self.patch_size + 1, self.stride):
                box = (x, y, x + self.patch_size, y + self.patch_size)
                patch = image.crop(box)
                yield {
                    "image": patch,
                    "bbox": (x, y, self.patch_size, self.patch_size),
                    "index": (x_idx, y_idx)
                }
                x_idx += 1
            y_idx += 1

    def reconstruct_image(self, patches, bboxes, original_size):
        width, height = original_size
        first_patch = patches[0]
        channels = first_patch.shape[2] if len(first_patch.shape) > 2 else 1
        
        recon_shape = (height, width, channels) if channels > 1 else (height, width)
        recon_img = np.zeros(recon_shape, dtype=np.float32)
        count_img = np.zeros((height, width), dtype=np.float32)
        
        for patch, bbox in zip(patches, bboxes):
            x, y, w, h = bbox
            if len(patch.shape) == 3 and patch.shape[2] == 1:
                patch = patch.squeeze(axis=2)
                
            if channels > 1:
                recon_img[y:y+h, x:x+w, :] += patch
            else:
                recon_img[y:y+h, x:x+w] += patch
            count_img[y:y+h, x:x+w] += 1.0
            
        count_mask = count_img > 0
        if channels > 1:
            for c in range(channels):
                recon_img[:, :, c][count_mask] /= count_img[count_mask]
        else:
            recon_img[count_mask] /= count_img[count_mask]
            
        return np.clip(recon_img, 0, 255).astype(np.uint8)

In [ ]:
class MockPredictor:
    def __init__(self):
        self.classes = [
            "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
            "Industrial", "Pasture", "PermanentCrop", "Residential",
            "River", "SeaLake"
        ]
        
    def predict_detailed(self, images):
        pred_classes = []
        conf_scores = []
        prob_dists = []
        
        for img in images:
            probs = np.random.dirichlet(np.ones(10))
            pred_idx = np.argmax(probs)
            
            pred_classes.append(self.classes[pred_idx])
            conf_scores.append(float(probs[pred_idx]))
            prob_dists.append(probs.tolist())
            
        return pred_classes, conf_scores, prob_dists

In [ ]:
class LandCoverMapper:
    COLOR_MAP = {
        "AnnualCrop": [240, 150, 150],
        "Forest": [0, 128, 0],
        "HerbaceousVegetation": [150, 240, 150],
        "Highway": [128, 128, 128],
        "Industrial": [255, 0, 0],
        "Pasture": [255, 255, 0],
        "PermanentCrop": [200, 100, 50],
        "Residential": [255, 165, 0],
        "River": [0, 0, 255],
        "SeaLake": [0, 255, 255]
    }
    
    def __init__(self, predictor, patch_size=64, stride=64):
        self.predictor = predictor
        self.patch_size = patch_size
        self.stride = stride
        self.patch_generator = PatchGenerator(patch_size, stride)
        
    def generate_map(self, original_img: Image.Image, batch_size=4):
        width, height = original_img.size
        patches = []
        bboxes = []
        
        for patch_data in self.patch_generator.extract_patches(original_img):
            patches.append(patch_data["image"])
            bboxes.append(patch_data["bbox"])
            
        predictions = []
        confidences = []
        for i in range(0, len(patches), batch_size):
            batch_patches = patches[i:i+batch_size]
            preds, confs, _ = self.predictor.predict_detailed(batch_patches)
            predictions.extend(preds)
            confidences.extend(confs)
            
        color_patches = []
        conf_patches = []
        for pred, conf in zip(predictions, confidences):
            color = self.COLOR_MAP.get(pred, [0, 0, 0])
            cp = np.zeros((self.patch_size, self.patch_size, 3), dtype=np.uint8)
            cp[:, :, :] = color
            color_patches.append(cp)
            
            cfp = np.zeros((self.patch_size, self.patch_size, 1), dtype=np.uint8)
            cfp[:, :, 0] = int(conf * 255)
            conf_patches.append(cfp)
            
        recon_color = self.patch_generator.reconstruct_image(color_patches, bboxes, (width, height))
        recon_conf = self.patch_generator.reconstruct_image(conf_patches, bboxes, (width, height))
        
        return {
            "prediction_map": Image.fromarray(recon_color),
            "confidence_map": Image.fromarray(recon_conf.squeeze(), mode="L"),
            "original_image": original_img
        }

In [ ]:
original_img = Image.new("RGB", (256, 256), color=(34, 139, 34))

predictor = MockPredictor()
mapper = LandCoverMapper(predictor, patch_size=64, stride=64)
outputs = mapper.generate_map(original_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(outputs["original_image"])
axes[0].set_title("Original Sentinel-2 Mock")
axes[0].axis("off")

axes[1].imshow(outputs["prediction_map"])
axes[1].set_title("Reconstructed Land Cover")
axes[1].axis("off")

axes[2].imshow(outputs["confidence_map"], cmap="gray")
axes[2].set_title("Confidence Heatmap")
axes[2].axis("off")

plt.tight_layout()
plt.show()